In [20]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv(r"C:\Users\hamen\OneDrive\Desktop\projects set\project5\dataset\swiggy_raw_dataset.csv")
df.head()

,order_id,customer_id,city,restaurant,cuisine,order_amount,delivery_time_mins,rating,order_status,payment_mode,order_date
0,40,1474,Mumbai,Biryani House,North Indian,709.13,28,4.7,Pending,Wallet,2024-01-04
1,1603,1936,Bangalore,Spice Hub,North Indian,938.23,40,3.6,Pending,Wallet,2024-02-02
2,3344,1252,Mumbai,Pizza World,Desserts,470.62,43,4.7,Cancelled,UPI,2024-01-17
3,4549,1419,Mumbai,Biryani House,Desserts,1448.17,23,4.2,Pending,Wallet,2024-04-22
4,3610,1630,Hyderabad,Urban Tadka,NaN,567.30,10,3.4,Delivered,Wallet,2024-09-05


Drop duplicates values


In [3]:
df = df.drop_duplicates()
df['order_id'].duplicated().sum()

np.int64(0)

Fixed datatype of order_Date


In [4]:
df['order_date'] = pd.to_datetime(df['order_date'])
df.info()

<class 'pandas.DataFrame'>
Index: 5000 entries, 0 to 5248
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   order_id            5000 non-null   int64         
 1   customer_id         5000 non-null   int64         
 2   city                4750 non-null   str           
 3   restaurant          4750 non-null   str           
 4   cuisine             4750 non-null   str           
 5   order_amount        5000 non-null   float64       
 6   delivery_time_mins  5000 non-null   int64         
 7   rating              4750 non-null   float64       
 8   order_status        5000 non-null   str           
 9   payment_mode        5000 non-null   str           
 10  order_date          5000 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(2), int64(3), str(5)
memory usage: 468.8 KB


separate columns on the basis of there datatypes

In [11]:
df_col = df[['order_id','customer_id','order_date']]
num_cols =  ['order_amount','delivery_time_mins','rating']
cat_cols = ['city','restaurant','cuisine','order_status','payment_mode']

create pipline

In [6]:
num_pipline = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median'))
    ]
)
cat_pipline = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='constant',fill_value='unknown'))
    ]
)

In [7]:
preprocessing = ColumnTransformer(transformers=[
    ('num',num_pipline,num_cols),
    ('cat',cat_pipline,cat_cols)
])

Apply pipline

In [12]:
df_features = df.drop(columns=['order_id', 'order_date'])


df_clean = pd.DataFrame(
    preprocessing.fit_transform(df_features),
    columns=preprocessing.get_feature_names_out()
)

df_final = pd.concat([
    df_col.reset_index(drop=True),
    df_clean.reset_index(drop=True)
], axis=1)
df_final

,order_id,customer_id,order_date,num__order_amount,num__delivery_time_mins,num__rating,cat__city,cat__restaurant,cat__cuisine,cat__order_status,cat__payment_mode
0,40,1474,2024-01-04,709.13,28.0,4.7,Mumbai,Biryani House,North Indian,Pending,Wallet
1,1603,1936,2024-02-02,938.23,40.0,3.6,Bangalore,Spice Hub,North Indian,Pending,Wallet
2,3344,1252,2024-01-17,470.62,43.0,4.7,Mumbai,Pizza World,Desserts,Cancelled,UPI
3,4549,1419,2024-04-22,1448.17,23.0,4.2,Mumbai,Biryani House,Desserts,Pending,Wallet
4,3610,1630,2024-09-05,567.3,10.0,3.4,Hyderabad,Urban Tadka,unknown,Delivered,Wallet
...,...,...,...,...,...,...,...,...,...,...,...
4995,221,1573,2024-07-25,1200.82,21.0,1.8,Mumbai,Sushi Zen,North Indian,Cancelled,Cash
4996,4464,1652,2024-02-11,124.17,13.0,2.0,Chennai,Biryani House,Desserts,Pending,Cash
4997,4501,1550,2024-04-02,334.96,26.0,1.7,unknown,Spice Hub,Chinese,Pending,UPI
4998,3959,1983,2024-05-26,109.23,59.0,1.6,Mumbai,Sushi Zen,Fast Food,Delivered,Cash


check new clean dataset

In [13]:
df_final.isnull().sum()

order_id                   0
customer_id                0
order_date                 0
num__order_amount          0
num__delivery_time_mins    0
num__rating                0
cat__city                  0
cat__restaurant            0
cat__cuisine               0
cat__order_status          0
cat__payment_mode          0
dtype: int64

In [16]:
df_final.duplicated().sum()

np.int64(0)

In [19]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   order_id                 5000 non-null   int64         
 1   customer_id              5000 non-null   int64         
 2   order_date               5000 non-null   datetime64[us]
 3   num__order_amount        5000 non-null   object        
 4   num__delivery_time_mins  5000 non-null   object        
 5   num__rating              5000 non-null   object        
 6   cat__city                5000 non-null   str           
 7   cat__restaurant          5000 non-null   str           
 8   cat__cuisine             5000 non-null   str           
 9   cat__order_status        5000 non-null   str           
 10  cat__payment_mode        5000 non-null   str           
dtypes: datetime64[us](1), int64(2), object(3), str(5)
memory usage: 429.8+ KB


Make new columns like year,month

In [31]:
df_final['months'] = df_final['order_date'].dt.month_name()
df_final['years'] = df_final['order_date'].dt.year
df_final['years']

0       2024
1       2024
2       2024
3       2024
4       2024
        ... 
4995    2024
4996    2024
4997    2024
4998    2024
4999    2024
Name: years, Length: 5000, dtype: int32

In [32]:
df_final.head()

,order_id,customer_id,order_date,num__order_amount,num__delivery_time_mins,num__rating,cat__city,cat__restaurant,cat__cuisine,cat__order_status,cat__payment_mode,months,years
0,40,1474,2024-01-04,709.13,28.0,4.7,Mumbai,Biryani House,North Indian,Pending,Wallet,January,2024
1,1603,1936,2024-02-02,938.23,40.0,3.6,Bangalore,Spice Hub,North Indian,Pending,Wallet,February,2024
2,3344,1252,2024-01-17,470.62,43.0,4.7,Mumbai,Pizza World,Desserts,Cancelled,UPI,January,2024
3,4549,1419,2024-04-22,1448.17,23.0,4.2,Mumbai,Biryani House,Desserts,Pending,Wallet,April,2024
4,3610,1630,2024-09-05,567.3,10.0,3.4,Hyderabad,Urban Tadka,unknown,Delivered,Wallet,September,2024


Convert csv clean data into sqlite table

In [33]:
conn = sqlite3.connect(r"C:\Users\hamen\OneDrive\Desktop\projects set\project5\swiggy_database.db")

In [34]:
df_final.to_sql('swaggy_db', conn, if_exists='replace', index=False)

5000

lets check database

In [35]:
q1 = 'select*from swaggy_db;'
d1 = pd.read_sql(q1,conn)
d1

,order_id,customer_id,order_date,num__order_amount,num__delivery_time_mins,num__rating,cat__city,cat__restaurant,cat__cuisine,cat__order_status,cat__payment_mode,months,years
0,40,1474,2024-01-04 00:00:00,709.13,28.0,4.7,Mumbai,Biryani House,North Indian,Pending,Wallet,January,2024
1,1603,1936,2024-02-02 00:00:00,938.23,40.0,3.6,Bangalore,Spice Hub,North Indian,Pending,Wallet,February,2024
2,3344,1252,2024-01-17 00:00:00,470.62,43.0,4.7,Mumbai,Pizza World,Desserts,Cancelled,UPI,January,2024
3,4549,1419,2024-04-22 00:00:00,1448.17,23.0,4.2,Mumbai,Biryani House,Desserts,Pending,Wallet,April,2024
4,3610,1630,2024-09-05 00:00:00,567.30,10.0,3.4,Hyderabad,Urban Tadka,unknown,Delivered,Wallet,September,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,221,1573,2024-07-25 00:00:00,1200.82,21.0,1.8,Mumbai,Sushi Zen,North Indian,Cancelled,Cash,July,2024
4996,4464,1652,2024-02-11 00:00:00,124.17,13.0,2.0,Chennai,Biryani House,Desserts,Pending,Cash,February,2024
4997,4501,1550,2024-04-02 00:00:00,334.96,26.0,1.7,unknown,Spice Hub,Chinese,Pending,UPI,April,2024
4998,3959,1983,2024-05-26 00:00:00,109.23,59.0,1.6,Mumbai,Sushi Zen,Fast Food,Delivered,Cash,May,2024
